kernel : Python (Pixi)

#### Bioproject: PRJNA1108209 & PRJNA694996 (Reference Genome: calJac240_pri) 

In [ ]:
%load_ext autoreload
%autoreload 2
import anndata as ad
import gc
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir

merged_h5ad_dir = find_env_dir("MERGED_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

In [ ]:
EAE_FILE = "PRJNA1108209.h5ad"
SERIES_NAME = "lin"
SAVE_FILE_NAME = SERIES_NAME + "_raw.h5ad"
eae_adata = sc.read_h5ad(os.path.join(merged_h5ad_dir, EAE_FILE))
eae_adata.obs.head() #type: ignore

,run,source_name,tissue,cell line,cell type,il01 uniqueid,il02 species,il03 source,il04 sex,il05 agedays,...,il08 condition,il09 illumina,il10 chemistry,il11 batch,il12 lmindays,il13 lmaxdays,il14 dataset,geo_loc_name,collection_date,sample
TTCACGCGTACTCCCT-1-SRR28906683,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,F,2456,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
AAACCCACAATAACCC-1-SRR28906683,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,F,2456,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
ATCACGATCTCGGCTT-1-SRR28906683,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,F,2456,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
GTTGTGAAGCGAATGC-1-SRR28906683,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,F,2456,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
GCCAACGTCCGTAATG-1-SRR28906683,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,F,2456,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397


In [ ]:
assert isinstance(eae_adata.obs, pd.DataFrame)
eae_adata.obs["run"] = eae_adata.obs["run"]
eae_adata.obs["matter"] = eae_adata.obs["tissue"]
eae_adata.obs["donor"] = eae_adata.obs["il03 source"]
eae_adata.obs["sex"] = eae_adata.obs["il04 sex"]
eae_adata.obs["age"] = eae_adata.obs["il05 agedays"].astype(float)
eae_adata.obs["location"] = eae_adata.obs["il07 location"].str.split('.', n=1).str[1]
eae_adata.obs["condition"] = eae_adata.obs["il08 condition"]
eae_adata.obs["batch"] = eae_adata.obs["il11 batch"]
eae_adata.obs["lesion_age_min"] = eae_adata.obs["il12 lmindays"].astype(float)
eae_adata.obs["lesion_age_max"] = eae_adata.obs["il13 lmaxdays"].astype(float)
eae_adata.obs["sample"] = eae_adata.obs["sample"]
eae_adata.obs["series"] = SERIES_NAME

age_mean = eae_adata.obs["age"].mean()
age_sd = eae_adata.obs["age"].std()
eae_adata.obs["age_scale"] = (eae_adata.obs["age"] - age_mean) / (2 * age_sd)

keep = ["run", "matter", "donor", "sex", "age_scale", "location", "condition", "batch", "lesion_age_min", "lesion_age_max", "sample", "series"]
eae_adata.obs.drop(columns = [c for c in eae_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
CONTROL_FILE = "PRJNA694996.h5ad"
SERIES_NAME = "lin"
control_adata = sc.read_h5ad(os.path.join(merged_h5ad_dir, CONTROL_FILE))
control_adata.obs.head() #type: ignore

In [ ]:
assert isinstance(control_adata.obs, pd.DataFrame)


In [ ]:
if set(eae_adata.obs.columns) != set(control_adata.obs.columns):
    raise ValueError("obs of eae_adata and control_adata do not match!")

adata = ad.concat([eae_adata, control_adata], join="outer")

In [11]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE_NAME))
del adata
gc.collect()

5224